In [ ]:
# Installing and updating dependencies
!pip install -U peft bitsandbytes transformers accelerate # accelerate - for multi GPU setup
!pip install -U trl # transformer - reinforcement learning
!pip install PyMuPDF # handling pdf files

In [3]:
# Downloading the dataset
from datasets import Dataset, load_dataset
dataset = load_dataset("roneneldan/TinyStories", split="train")
print(dataset[0]['text'])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [2]:
# Loading data from PDF
import fitz

def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc: # Load PDF pagewise
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

pdf_texts = extract_text_from_pdf("/content/Metformin.pdf")
print(pdf_texts[0])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

In [3]:
import re

# Splitting paragraph wise
def split_paragraphs(pages):
    paragraphs = []

    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)

        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:  # ignore too short lines
                paragraphs.append(clean)

    return paragraphs

paragraphs = split_paragraphs(pdf_texts)
data = [{"text": p} for p in paragraphs]
data

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'},
 {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits h

In [6]:
# Converting to Hugging Face Dataset object
dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

---

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Downloading tokenizer
model_name = "sandeep1121/unsloth_instruction_finetuning_tinyllama_adapter"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Adding <EOS> token at the end of every document
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [9]:
# Tokenizing our dataset
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [10]:
import torch
# from transformers import BitsAndBytesConfig

# Define the quantization configuration
"""
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
"""

# Load the model with quantization
model_id = "sandeep1121/unsloth_instruction_finetuning_tinyllama_adapter"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    # device_map="auto"
    # quantization_config=quant_config,
)

model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/176 [00:00<?, ?it/s]

In [ ]:
from transformers import TrainingArguments
help(TrainingArguments)

In [11]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to GPU
model.to(device)

# Set format for training, but don't use .to(device) on the dataset itself
tokenized.set_format("torch")

In [12]:
training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=False,      # Disabled to fix the AssertionError
    bf16=torch.cuda.is_bf16_supported(), # Use BF16 if available, else FP32
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

In [13]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2, training_loss=6.557605743408203, metrics={'train_runtime': 15.8063, 'train_samples_per_second': 0.506, 'train_steps_per_second': 0.127, 'total_flos': 25534905974784.0, 'train_loss': 6.557605743408203, 'epoch': 2.0})

---

In [ ]:
# Partial Fine Tuning -> Feerze some layer and finetune unfreeze layer(old CNN and Bert sytel method)

# -------------------------------------------------------------
# 1️⃣ Install dependencies
# -------------------------------------------------------------
!pip install -q transformers datasets accelerate bitsandbytes

# -------------------------------------------------------------
# 2️⃣ Imports
# -------------------------------------------------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import os, re

# -------------------------------------------------------------
# 3️⃣ Load base model and tokenizer
# -------------------------------------------------------------
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# -------------------------------------------------------------
# 4️⃣ Freeze everything
# -------------------------------------------------------------
for param in model.parameters():
    param.requires_grad = False

# -------------------------------------------------------------
# 5️⃣ Unfreeze last 4 transformer blocks + lm_head
# -------------------------------------------------------------
for name, param in model.named_parameters():
    if any(f"layers.{i}." in name for i in range(20, 24)):  # last 4 layers
        param.requires_grad = True
    if "lm_head" in name:
        param.requires_grad = True

# Verify how many parameters are trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable: {trainable/total*100:.2f}% of total parameters")

# -------------------------------------------------------------
# 6️⃣ Load your dataset
# -------------------------------------------------------------
dataset = load_dataset("json", data_files={"train": "pharma_non_instruction.jsonl"})

def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset["train"].map(tokenize_fn, batched=True, remove_columns=["text"])

# -------------------------------------------------------------
# 7️⃣ Data collator for causal LM
# -------------------------------------------------------------
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# -------------------------------------------------------------
# 8️⃣ Training arguments
# -------------------------------------------------------------
os.environ["WANDB_DISABLED"] = "true"  # disable wandb
training_args = TrainingArguments(
    output_dir="./tinyllama-pharma-last4",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,        # slightly higher since fewer params
    fp16=True,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    report_to="none"
)

# -------------------------------------------------------------
# 9️⃣ Trainer setup
# -------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator
)

# -------------------------------------------------------------
# 🔟 Start training
# -------------------------------------------------------------
trainer.train()

# -------------------------------------------------------------
# 11️⃣ Save final model
# -------------------------------------------------------------
trainer.save_model("./tinyllama-pharma-last4-final")
print("\n✅ Training completed and model saved!")


### Datasets

#### Non-Instruction Finetuning Dataset
- HuggingFaceFW/fineweb
- ncbi/pubmed
- datajuicer/the-pile-pubmed-abstracts-refined-by-data-juicer
- open-llm-leaderboard/open_llm_corpus
- Skylion007/openwebtext
- armanc/scientific_papers

#### Instruction Finetuning Dataset
- Amod/mental_health_counseling_conversations
- yahma/alpaca-cleaned
- Open-Orca/OpenOrca
- tatsu-lab/alpaca
- OpenAssistant/oasst1

#### Prefrence Alignemnt tuing dataset
- Anthropic/hh-rlhf
- argilla/ultrafeedback-binarized-preferences-cleaned
- xinlai/Math-Step-DPO-10K


---

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [15]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [16]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [17]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [18]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe reduced coat weiterclosurechus reactjs экс «Milcian pace reactjschus coat thy coat mars jak weiter seule ressolate shapes say:|framework/components эксни Людиимель dess underlying route coatown lettкт « célณ���������� «андро быть suivantes angularjs structures permissions and machiami Roman inclusiversiedΈณ осуще����正学станlongrightarrow jaknée pitt�����������えURI nucpol


In [19]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

combined_dataset.json:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [20]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [21]:
def format_row(example):
    question = example["Context"]
    answer = example["Response"]
    example["Text"] = f"[Context] {question} [/Response] {answer}"
    return example

formatted_dataset = dataset.map(format_row)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [22]:
formatted_dataset

Dataset({
    features: ['Context', 'Response', 'Text'],
    num_rows: 3512
})

In [23]:
print(formatted_dataset[0]["Text"])

[Context] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone? [/Response] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media.  Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.

In [24]:
import pandas as pd

# Convert dataset to DataFrame
df = pd.DataFrame(dataset)

df.to_csv("mental_health_counseling_conversations.csv", index=False)
df.to_json("mental_health_counseling_conversations.jsonl", orient="records", lines=True)

In [25]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="mental_health_counseling_conversations.csv", split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [26]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="mental_health_counseling_conversations.jsonl", split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [28]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [29]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

dataset = dataset.map(format_example)
dataset

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 5
})

In [30]:
dataset['text'][0]

'### Instruction:\nExplain the mechanism of action of Metformin.\n### Input:\nNone\n### Response:\nMetformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'

In [31]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [32]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

| Parameter        | Meaning                     | Typical Value         | Effect                           |
| ---------------- | --------------------------- | --------------------- | -------------------------------- |
| `task_type`      | Model type (Causal/Seq2Seq) | `CAUSAL_LM`           | Ensures correct integration      |
| `r`              | Rank of LoRA matrix         | 4–16                  | Controls trainable param size    |
| `lora_alpha`     | Scaling factor              | 16–64                 | Balances adaptation strength     |
| `lora_dropout`   | Dropout probability         | 0.05                  | Regularization                   |
| `target_modules` | Which layers to tune        | `["q_proj","v_proj"]` | Trade-off between cost & quality |
| `bias`           | Bias fine-tuning            | `"none"`              | Keep simple                      |


In [33]:
instruction_model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [35]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=instruction_model,
    args=args,
    train_dataset=tokenized,
)

In [36]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=3, training_loss=6.698144912719727, metrics={'train_runtime': 26.9055, 'train_samples_per_second': 0.558, 'train_steps_per_second': 0.112, 'total_flos': 47826044190720.0, 'train_loss': 6.698144912719727, 'epoch': 3.0})

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

---

In [ ]:
# The instructions get masked with the spaecial tokens - another way
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Load dataset
dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")
print(dataset)


# Format dataset to Alpaca-style text
def format_example(example):
    # Build unified instruction-style prompt
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

dataset = dataset.map(format_example)
print(dataset[0]["text"])


# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Tokenization with Response Masking
def tokenize_and_mask(example):
    text = example["text"]

    # Tokenize full text
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
    input_ids = enc["input_ids"]

    # Find where '### Response:' starts
    response_marker = "### Response:"
    response_start = text.find(response_marker)

    if response_start != -1:
        # Token index where response begins
        response_token_start = len(tokenizer(text[:response_start])["input_ids"])
    else:
        response_token_start = 0  # if marker not found

    # Clone labels and mask out everything before 'Response'
    labels = input_ids.copy()
    labels[:response_token_start] = [-100] * response_token_start

    enc["labels"] = labels
    return enc

# Apply tokenization
tokenized = dataset.map(tokenize_and_mask, batched=False)
print("Tokenization + masking done.")


# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)


# Load base model (previously non-instructional trained)
non_instructional_trained_model = AutoModelForCausalLM.from_pretrained(
    "path_to_your_non_instruction_model",
    torch_dtype=torch.float16,
    device_map="auto"
)

model = get_peft_model(non_instructional_trained_model, lora_config)

# Training setup
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)


# Train the model
trainer.train()


# Save & test the model
trainer.save_model("/content/tinyllama-instruction")
tokenizer.save_pretrained("/content/tinyllama-instruction")

# Test generation
trained_model = AutoModelForCausalLM.from_pretrained("/content/tinyllama-instruction", device_map="auto")

prompt = "### Instruction:\nWhat is Ezetimibe?\n### Input:\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
questions = [
    "Explain the mechanism of action of Metformin.",
    "List two advantages of combining Atorvastatin with Ezetimibe.",
    "Summarize how mRNA vaccines work and mention one current research focus."
]

In [ ]:
for q in questions:
    print("Question:", q)
    print("\n--- Non-instruction model ---")
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = non_instruction_model.generate(**inputs, max_new_tokens=80)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = instruction_model.generate(**inputs, max_new_tokens=100)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("="*80, "\n")